# Train YOLO26 on TiBLINED

Fine-tune a line detector and a semantic segmentation model on the frozen TiBLINED splits. The detector predicts one box per annotated line. The semantic model predicts a foreground mask covering all annotated text lines.

Run this notebook from the repository root with a Python kernel that has this repo's dependencies and a PyTorch build for your device. The local Parquet snapshot is used if present. For a new copy of this private dataset, run `hf auth login` in a terminal with a Hugging Face account that has access.

Dataset and annotations by [BDRC / BUDA](https://huggingface.co/datasets/BDRC/TiBLINED). [Ultralytics YOLO26](https://docs.ultralytics.com/models/yolo26/)

In [ ]:
import subprocess
import sys
from collections import deque
from pathlib import Path

SOURCE = Path("data/TiBLINED")
PREPARED = Path("data/tiblined_yolo")
PROJECT = Path("runs/tiblined")
DEVICE = None  # None lets Ultralytics choose, or use "0", "cpu", or "mps".
IMG_SIZE = 1024
BATCH = 4
WORKERS = 0  # Start with 0 for notebook and Windows compatibility.
EPOCHS = 100
SEED = 0
DETECT_MODEL = "yolo26n.pt"
SEMANTIC_MODEL = "yolo26n-sem.pt"
EVAL_TEST = False  # Enable once settings are finalized using validation results.
SMOKE_PAGES = None  # Set to 8 for a one-epoch check on a small subset.

if SMOKE_PAGES is not None:
    PREPARED = PREPARED.with_name(f"{PREPARED.name}_smoke_{SMOKE_PAGES}")
    PROJECT = PROJECT / f"smoke_{SMOKE_PAGES}"
    EPOCHS = 1


def run_script(script, *args):
    command = [sys.executable, "-X", "utf8", "-u", script, *map(str, args)]
    tail = deque(maxlen=12)
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        encoding="utf-8",
        errors="replace",
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                tail.append(line)
        except KeyboardInterrupt:
            process.kill()
            raise
        if exit_code := process.wait():
            details = "".join(tail).strip() or "The script produced no output."
            raise RuntimeError(f"{script} exited with status {exit_code}.\n{details}")

## Download and prepare

The converter streams the embedded scans and PAGE-XML. It keeps the published train, validation, and test assignments and writes YOLO detection labels and pixel masks at the original image size. The conversion reports repaired, clipped, and dropped annotations.

In [ ]:
prepared_splits = ("train", "val", "test")


def is_prepared(path):
    return all(
        (path / f"{task}.yaml").is_file() for task in ("detect", "semantic")
    ) and all(
        any(item.is_file() for item in (path / folder / split).glob(pattern))
        for folder, pattern in (
            ("images", "*.*"),
            ("labels", "*.txt"),
            ("masks", "*.png"),
        )
        for split in prepared_splits
    )


requested_output = PREPARED
attempt = 1
while PREPARED.exists() and not is_prepared(PREPARED):
    PREPARED = requested_output.with_name(f"{requested_output.name}_rebuild_{attempt}")
    attempt += 1
if PREPARED != requested_output:
    print(f"Incomplete export at {requested_output}. Using {PREPARED}.")

if not is_prepared(PREPARED):
    source_splits = ("train", "validation", "test")
    if not all(
        (SOURCE / "data" / f"{split}.parquet").is_file() for split in source_splits
    ):
        run_script("download_tiblined.py", "--output", SOURCE)

    prepare_args = ["--source", SOURCE, "--output", PREPARED]
    if SMOKE_PAGES is not None:
        prepare_args.extend(["--limit-per-split", SMOKE_PAGES])
    run_script("prepare_tiblined.py", *prepare_args)

print(f"Using prepared dataset at {PREPARED.resolve()}")
for split in prepared_splits:
    print(
        f"{split}: {sum(p.is_file() for p in (PREPARED / 'images' / split).iterdir())} pages"
    )

## Inspect a prepared page

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
from PIL import Image

image_path = min(p for p in (PREPARED / "images" / "train").iterdir() if p.is_file())
with Image.open(image_path) as image:
    scan = image.convert("RGB")
with Image.open(PREPARED / "masks" / "train" / f"{image_path.stem}.png") as image:
    mask = np.array(image)
width, height = scan.size
label_file = PREPARED / "labels" / "train" / f"{image_path.stem}.txt"

fig, (boxes_ax, mask_ax) = plt.subplots(2, 1, figsize=(13, 8))
for ax in (boxes_ax, mask_ax):
    ax.imshow(scan)
    ax.axis("off")

for label in label_file.read_text(encoding="utf-8").splitlines():
    _, cx, cy, w, h = map(float, label.split())
    origin = ((cx - w / 2) * width, (cy - h / 2) * height)
    boxes_ax.add_patch(
        Rectangle(origin, w * width, h * height, fill=False, edgecolor="lime")
    )

boxes_ax.set_title("Line boxes")
mask_ax.imshow(np.ma.masked_equal(mask, 0), cmap="spring", vmin=0, vmax=1, alpha=0.45)
mask_ax.set_title("Line foreground mask")
plt.tight_layout()

## Train the two models

Run the training cells independently. Ultralytics saves each run's validation metrics and checkpoints under `PROJECT`. Enable `EVAL_TEST` to evaluate the best validation checkpoint on the held-out test split. Detection reports box mAP. Semantic segmentation reports pixel metrics including mean IoU.

In [ ]:
def train_task(task, model):
    args = [
        "--task",
        task,
        "--model",
        model,
        "--data",
        PREPARED / f"{task}.yaml",
        "--img-size",
        IMG_SIZE,
        "--epochs",
        EPOCHS,
        "--batch",
        BATCH,
        "--workers",
        WORKERS,
        "--seed",
        SEED,
        "--project",
        PROJECT,
    ]
    if DEVICE is not None:
        args.extend(["--device", DEVICE])
    if EVAL_TEST:
        args.append("--eval-test")
    run_script("train.py", *args)

In [ ]:
train_task("detect", DETECT_MODEL)

In [ ]:
train_task("semantic", SEMANTIC_MODEL)